# Topic modeling · data besar (ratusan ribu baris)
**Versi skala besar dari `01_topic_modeling_review`. Target: Run All di Kaggle T4 jauh di bawah 5 jam.**

| Masalah di data besar | Solusi |
|---|---|
| Banyak teks identik | Model jalan di **teks unik** `U`, dengan `count` sebagai bobot. Ukuran topik dihitung per baris asli |
| Grid model × k × seed mahal | Pemilihan dilakukan di **sampel `N_FIT` teks unik** |
| Fit ulang di semua data mahal | Model dari sampel diterapkan ke semua teks: `transform` untuk LDA/NMF, `predict` untuk KMeans, dan centroid e5 untuk HDBSCAN. Kecocokan sampel dengan hasil penuh dicek di bagian 09 |
| Stemming lambat | Hanya kata yang lolos `MIN_DF`, 4 proses `forkserver` |

**Model** sama dengan versi review: acak, TF-IDF + KMeans, LDA, NMF, e5 + KMeans, dan e5 + UMAP + HDBSCAN (gaya BERTopic). Backbone e5 di-freeze.

**Aturan pemilihan tanpa label** juga sama: topik raksasa dan outlier berlebihan dilarang, lalu kandidat diurutkan dengan rata-rata peringkat TQ (coherence × diversity) dan stabilitas antar seed. Di versi review, stabilitas adalah kriteria yang paling searah dengan audit (Spearman +0,60), sedangkan TQ +0,19.

**Data:** `news` = 91 ribu judul berita, kategori sebagai audit. `gojek` = 225 ribu ulasan aplikasi, tanpa label topik.

## 00 · Konfigurasi

Parameter papermill. `audit` diisi `None` kalau labelnya tidak ada. `desc` adalah kolom angka opsional untuk membaca topik, misalnya rating.

In [1]:
DATASET = "news"
SEED, TOP = 42, 10
N_ROWS, N_FIT = None, 30000
K_GRID = [8, 10, 15, 20, 25, 30]
SEEDS = 3
MAX_SHARE, MAX_OUTLIER = .40, .30
MIN_DF, DF_MAX, MIN_TOPIC_SHARE, MERGE_JACCARD, GIANT_GENERIC = 5, .25, .005, .30, .25
MAX_LEN, BATCH = 128, 256
BUDGET_MINUTES = 290
PRESETS = {"news": dict(file="indonesian-news-title.csv", text="title", audit="category", desc=None),
           "gojek": dict(file="GojekAppReviewV4.0.0-V4.9.3_Cleaned.csv", text="content", audit=None, desc="score")}

In [2]:
# Parameters
DATASET = "news"


In [3]:
from pathlib import Path
P = PRESETS[DATASET]; FILE, TEXT_COL, AUDIT_COL, DESC_COL = P["file"], P["text"], P["audit"], P["desc"]
OUT = Path(f"/kaggle/working/hasil_topik_besar_{DATASET}")
print("dataset:", DATASET, "| teks:", TEXT_COL, "| audit:", AUDIT_COL or "tidak ada")

dataset: news | teks: title | audit: category


## 01 · Lingkungan Kaggle & pencatat waktu

Waktu tiap tahap dicatat, dan notebook dibuat gagal kalau total melewati `BUDGET_MINUTES`.

In [4]:
import sys, os, subprocess
assert Path("/kaggle/input").exists(), "Jalankan notebook ini di Kaggle."
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn>=0.5,<0.6", "gensim>=4.4,<5", "ftfy>=6.3,<7",
                "emoji>=2.14,<3", "langid==1.1.6", "Sastrawi==1.0.1"], check=True)
os.environ.update(TOKENIZERS_PARALLELISM="false", HF_HUB_DISABLE_PROGRESS_BARS="1", USE_TF="0", USE_FLAX="0")

In [5]:
import json, time, random, re, html, unicodedata, itertools, warnings
from collections import Counter
from contextlib import contextmanager
import numpy as np, pandas as pd, torch, emoji, ftfy
from IPython.display import display
from scipy.optimize import linear_sum_assignment
from scipy.stats import spearmanr
from sklearn.cluster import KMeans, HDBSCAN
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.preprocessing import normalize
warnings.filterwarnings("ignore")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT.mkdir(exist_ok=True); INPUT = Path("/kaggle/input"); RUN_START = time.perf_counter(); TIMES = []
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada")

@contextmanager
def stage(name, n=None):
    start = time.perf_counter(); yield
    TIMES.append({"tahap": name, "detik": time.perf_counter() - start, "n": n})
    print(f"[waktu] {name}: {TIMES[-1]['detik']:.0f} detik | total {(time.perf_counter() - RUN_START) / 60:.1f} menit")
def show(title, table):
    print("\n" + title); display((table.to_frame() if isinstance(table, pd.Series) else table).round(3))
def distribution(values):
    n = values.value_counts(dropna=False); return pd.DataFrame({"n": n, "%": n / (len(values) or np.nan) * 100})
pd.set_option("display.max_columns", None, "display.max_colwidth", 100, "display.width", 200)

GPU: Tesla T4


## 02 · Muat data & dedupe

Kualitas diaudit di semua baris. Model jalan di tabel `U` berisi teks unik. Label audit per teks unik diambil dari suara terbanyak, lalu disembunyikan sampai bagian 10.

In [6]:
with stage("muat_data"):
    path = next(INPUT.rglob(FILE)); raw = pd.read_csv(path, low_memory=False)
text = raw[TEXT_COL]
quality = pd.DataFrame({"teks_null": text.isna(), "teks_non_string": text.notna() & ~text.map(lambda x: isinstance(x, str)),
    "teks_kosong": text.fillna("").astype(str).str.strip().eq("")})
show("Kualitas input", pd.DataFrame({"n": quality.sum(), "%": quality.mean() * 100}))
data = raw.loc[~quality.any(axis=1)].reset_index(names="row_id")
if N_ROWS and len(data) > N_ROWS: data = data.sample(N_ROWS, random_state=SEED).reset_index(drop=True)
data["key"] = data[TEXT_COL].astype(str).str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()
agg = {"text": (TEXT_COL, "first"), "count": ("key", "size")}
if DESC_COL: agg["desc"] = (DESC_COL, "mean")
if AUDIT_COL: agg["_audit"] = (AUDIT_COL, lambda s: s.astype(str).mode().iloc[0])
U = data.groupby("key", sort=False).agg(**agg).reset_index(); W = U["count"].to_numpy()
show("Ukuran setelah dedupe", pd.Series({"baris": len(data), "teks_unik": len(U), "hemat_%": 100 * (1 - len(U) / len(data)), "teks_terbanyak_n": W.max()}))
show("Teks paling sering · kandidat topik palsu", U.nlargest(TOP, "count")[["text", "count"]])

[waktu] muat_data: 3 detik | total 0.1 menit

Kualitas input


,n,%
teks_null,0,0.0
teks_non_string,0,0.0
teks_kosong,0,0.0



Ukuran setelah dedupe


,0
baris,91017.000
teks_unik,90885.000
hemat_%,0.145
teks_terbanyak_n,7.000



Teks paling sering · kandidat topik palsu


,text,count
26025,Film yang Rilis Pekan Ini,7
40370,Banjir Jakarta,5
25678,5 Video Musik Pilihan Minggu Ini,4
472,IHSG Diperkirakan Melemah di Awal Pekan,3
26009,5 Lagu Baru Pilihan Minggu Ini,3
26399,5 Klip Baru Pilihan Minggu Ini,3
26428,5 Video Musik Baru Pilihan Minggu Ini,3
36416,Rekomendasi Buku Akhir Pekan Ini,3
54931,Jadwal Buka Puasa Hari Ini dan Imsak Besok untuk Medan Sekitarnya,3
60249,BMKG Prediksi Jakarta Cerah Berawan Hari Ini,3


## 03 · EDA wajib topic modeling · angka

Pertanyaannya sama dengan versi review: panjang dokumen, kosakata setelah pruning, kata di mana-mana, kolokasi, noise, dan bahasa. Semua dihitung berbobot baris.

In [7]:
with stage("eda", len(U)):
    t = U.text.astype(str); words = t.str.split().str.len().to_numpy()
    show("3.1 Panjang dokumen (berbobot baris)", pd.Series({"kata_median": np.median(np.repeat(words, W)), "<5_kata_%": 100 * W[words < 5].sum() / W.sum(),
        "<10_kata_%": 100 * W[words < 10].sum() / W.sum(), ">50_kata_%": 100 * W[words > 50].sum() / W.sum()}))
    seqs = t.str.casefold().str.findall(r"[^\W\d_]+")
    counts = pd.Series(Counter(w for s in seqs for w in s), dtype="int64").sort_values(ascending=False)
    docfreq = pd.Series(Counter(w for s in seqs for w in set(s)), dtype="int64").reindex(counts.index) / len(seqs)
    kept = docfreq[(docfreq * len(seqs) >= MIN_DF) & (docfreq <= DF_MAX)]
    show("3.2 Kosakata & pruning · teks unik", pd.Series({"tipe_unik": len(counts), "hapax_%_tipe": 100 * counts.eq(1).mean(),
        "kosakata_setelah_pruning": len(kept), "token_tersisa_%": 100 * counts.reindex(kept.index).sum() / counts.sum(), f"kata_df>{DF_MAX:.0%}": int(docfreq.gt(DF_MAX).sum())}))
    show("Top words", pd.DataFrame({"n": counts.head(15), "%_dokumen": 100 * docfreq.head(15)}))
    pairs = Counter(" ".join(s[i:i + 2]) for s in seqs for i in range(len(s) - 1)); total = counts.sum()
    colloc = pd.DataFrame([{"bigram": b, "n": n, "PMI": np.log(n * total / (counts[b.split()[0]] * counts[b.split()[1]]))} for b, n in pairs.items() if n >= 30])
    show("3.3 Kolokasi · kandidat frasa", colloc.sort_values("PMI", ascending=False).head(TOP) if len(colloc) else colloc)
    NOISE = {"url": r"https?://|www\.", "html": r"</?[a-zA-Z][^>]*>", "mention": r"@\w+", "hashtag": r"#\w+", "angka": r"\d",
             "huruf_ulang": r"([^\W\d_])\1{2,}", "non_ASCII": r"[^\x00-\x7f]", "slang_contoh": r"\b(?:gk|gak|ga|bgt|yg|udh|klo|jg|tdk)\b"}
    flags = pd.DataFrame({n: t.str.contains(p, flags=re.I) for n, p in NOISE.items()})
    show("3.4 Noise · % baris", (flags.mul(W, axis=0).sum() / W.sum() * 100).sort_values(ascending=False))


3.1 Panjang dokumen (berbobot baris)


,0
kata_median,9.000
<5_kata_%,0.749
<10_kata_%,53.271
>50_kata_%,0.000



3.2 Kosakata & pruning · teks unik


,0
tipe_unik,32485.000
hapax_%_tipe,34.508
kosakata_setelah_pruning,12723.000
token_tersisa_%,92.532
kata_df>25%,1.000



Top words


,n,%_dokumen
di,27386,28.801
corona,17234,18.950
ini,10122,11.094
yang,8669,9.497
dan,6783,7.394
ke,5592,6.079
tak,5096,5.539
dari,4695,5.128
jadi,4486,4.915
untuk,3721,4.084



3.3 Kolokasi · kandidat frasa


,bigram,n,PMI
1339,ferdian paleka,30,10.223
1599,hanan attaki,31,10.190
990,fadli zon,30,10.190
1311,eternal monarch,31,10.158
1336,ariana grande,32,10.158
727,deg degan,30,10.158
1545,marion jola,32,10.158
1200,agnez mo,34,10.098
1365,raul lemos,32,10.098
1556,jenita janet,34,10.098



3.4 Noise · % baris


,0
angka,27.607
huruf_ulang,0.393
hashtag,0.354
non_ASCII,0.269
slang_contoh,0.056
mention,0.023
url,0.002
html,0.000


[waktu] eda: 2 detik | total 0.4 menit


## 04 · Preprocessing dari universal

Fungsi berikut diambil dari `preprocessing_universal/stepwise.py`. BoW agresif dipakai LDA, NMF, dan TF-IDF, sedangkan e5 memakai teks utuh. Stemming dikerjakan sebelum GPU dipakai, supaya proses paralelnya aman.

In [8]:
SLANG = dict(zip(
    'gk ga gak nggak ngga tdk blm jgn yg dgn utk krn karna bgt tp udh sdh skrg gmn klo kalo jg bgs'.split(),
    'tidak tidak tidak tidak tidak tidak belum jangan yang dengan untuk karena karena banget tapi sudah sudah sekarang bagaimana kalau kalau juga bagus'.split()))
PROTECTED = set('mbg sppg bgn apbn apbd bpom bpk umkm allah massa jakarta bandung'.split())
NEGATIONS = set('tidak tak bukan belum jangan tanpa kurang nggak gak ga gk ora teu indak'.split())
AMBIGUOUS = set('sm dr bs pd lg org'.split())
VOCAB = set('bagus banget lama makanan makan bergizi gratis program keracunan anggaran korupsi sehat sekolah anak siswa pemerintah merata enak'.split()) | set(SLANG.values()) | PROTECTED
STOPWORDS = set('yang dan di ke dari untuk pada dengan ini itu atau juga saya kami kita mereka dia oleh sebagai dalam para serta adalah sebuah'.split()) - NEGATIONS - PROTECTED
REGIONAL = {'ora': 'tidak', 'teu': 'tidak', 'indak': 'tidak', 'apik': 'bagus'}
OBFUSCATION = {'k3r4cunan': 'keracunan', 'k3r4cun4n': 'keracunan',
               'ker@cunan': 'keracunan', 'koru*si': 'korupsi'}
# URL/email/account/hashtag are single tokens: normalization never enters them.
TOKEN = re.compile(r'https?://[^\s<>]+|www\.[^\s<>]+|[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}|(?<!\w)[@#][\w.]+|[^\W\d_]+', re.I)
WORD = re.compile(r'[^\W\d_]+')
URL = re.compile(r'https?://[^\s<>]+|www\.[^\s<>]+', re.I)


def map_words(text, transform):
    return TOKEN.sub(lambda m: transform(m[0]) if WORD.fullmatch(m[0]) else m[0], text)

In [9]:
def valid_text(value):
    # None/NaN/pd.NA/numbers/list are not converted to literal "nan"/"123".
    return value if isinstance(value, str) else ''

def repair_encoding(text):
    return ftfy.fix_text(text, unescape_html=False, uncurl_quotes=False,
                        fix_latin_ligatures=False, fix_character_width=False,
                        remove_control_chars=False, normalization=None)

def normalize_unicode(text):
    return unicodedata.normalize('NFC', text)  # retain emoji/local scripts/fullwidth

def remove_html(text):
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(text, 'html.parser')
    for tag in soup(['script', 'style']):
        tag.decompose()
    return soup.get_text(' ')

def decode_html_entities(text):
    return html.unescape(text)

def remove_invisible(text):
    text = text.translate(dict.fromkeys(map(ord, '\ufeff\u200b\u2060\x00'), None))
    return re.sub(r'[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]', ' ', text)

def normalize_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

In [10]:
def replace_urls(text):
    # Preserve terminal sentence punctuation; a heuristic, not a complete URL parser.
    return URL.sub(lambda m: ' URL ' + m[0][len(m[0].rstrip('.,!?;)')):], text)

def replace_emails(text):
    return re.sub(r'\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b', ' EMAIL ', text)

def replace_phones(text):
    return re.sub(r'(?<!\w)(?:\+62|62|0)[ -]?8(?:[ -]?\d){8,11}(?!\w)', ' PHONE ', text)

def replace_mentions(text):
    return re.sub(r'(?<![\w@])@[\w.]+', ' USER ', text)

def segment_hashtags(text, mapping=None):
    mapping = {} if mapping is None else mapping
    def split(match):
        word = match[1]
        # #tolakmbg stays tolakmbg unless an explicit reviewed map is supplied.
        return ' ' + mapping.get(word.casefold(), re.sub(
            r'(?<=[a-z])(?=[A-Z])|(?<=[A-Z])(?=[A-Z][a-z])', ' ', word)) + ' '
    return re.sub(r'(?<!\w)#(\w+)', split, text)

def lowercase(text):
    return text.casefold()

In [11]:
def normalize_slang(text, mapping=SLANG):
    return map_words(text, lambda w: mapping.get(w.casefold(), w)
                     if w.casefold() not in PROTECTED | AMBIGUOUS else w)

def normalize_elongation(text, vocab=VOCAB):
    def shorten(word):
        if word.casefold() in PROTECTED:
            return word
        candidates = [re.sub(r'([^\W\d_])\1{2,}', replacement, word, flags=re.I)
                      for replacement in (r'\1', r'\1\1')]
        return next((w for w in candidates if w.casefold() in vocab), word)
    return map_words(text, shorten)

def normalize_laughter(text):
    def replace(word):
        return 'wkwk' if re.fullmatch(r'(?:wk){2,}|w+k+w+k+[wk]*', word, re.I) else (
            'haha' if re.fullmatch(r'(?:ha){2,}h?', word, re.I) else word)
    return map_words(text, replace)

def normalize_reduplication(text, mapping=None):
    # A whitelist avoids turning grade S2, model GPT2, dates or quantities into words.
    mapping = {'anak2': 'anak-anak', 'orang2': 'orang-orang', 'teman2': 'teman-teman'} if mapping is None else mapping
    return re.sub(r'\b\w+2\b', lambda m: mapping.get(m[0].casefold(), m[0]), text)

In [12]:
def remove_emoji(text):
    return emoji.replace_emoji(text, replace=' ')

def remove_emoticons(text):
    return re.sub(r"(?<!\w)(?:[:;=8][\-^']?[)(/DPp]+|<3)(?!\w)", ' ', text)

def remove_numbers(text):
    # Opt-in destructive view: amounts/dates become incomplete, inspect before using.
    return re.sub(r'(?<!\w)\d+(?:[.,:/-]\d+)*(?!\w)', ' ', text)

def remove_punctuation(text):
    # Keep all scripts/letters/digits; punctuation becomes a SPACE, never concatenation.
    return re.sub(r'[^\w\s]|_', ' ', text)

def reduce_repeated_words(text):
    # Aggressive only: even "bagus bagus" can be expressive, so retain another view.
    return re.sub(r'\b(\w+)(?:\s+\1\b)+', r'\1', text, flags=re.I)

In [13]:
def stem_series(series):
    """Stem hanya kata yang lolos MIN_DF, 4 proses forkserver."""
    import multiprocessing as mp
    Path("stem_worker.py").write_text("from Sastrawi.Stemmer.StemmerFactory import StemmerFactory\n_s = None\n"
        "def stem(w):\n    global _s\n    _s = _s or StemmerFactory().create_stemmer()\n    return _s.stem(w)\n")
    if "." not in sys.path: sys.path.insert(0, ".")
    import stem_worker
    vocab = pd.Series(Counter(w for s in series.str.split() for w in set(s)))
    frequent = [w for w in vocab[vocab >= MIN_DF].index if w not in PROTECTED | NEGATIONS]
    with mp.get_context("forkserver").Pool(4) as pool:
        stems = dict(zip(frequent, pool.map(stem_worker.stem, frequent, chunksize=256)))
    print(f"stemming {len(frequent)} kata dari {len(vocab)} kata unik")
    return series.map(lambda x: " ".join(stems.get(w, w) for w in x.split()))
def chain(text, steps):
    for step in steps: text = step(text)
    return text
def apply(series, steps):
    cache = {s: chain(s, steps) for s in series.unique()}; return series.map(cache)
SAFE = [valid_text, repair_encoding, normalize_unicode, lambda s: remove_html(s) if "<" in s else s, decode_html_entities, remove_invisible, normalize_whitespace]
NORMAL = [segment_hashtags, lowercase, normalize_slang, normalize_elongation, normalize_laughter, normalize_reduplication]
CLEAN = [remove_emoji, remove_emoticons, remove_numbers, remove_punctuation, reduce_repeated_words, normalize_whitespace]
STOP = STOPWORDS | set("""akan ada adalah agar aja aku apa atau bahwa banyak baru bila bisa buat bukan cuma dah dalam dapat demi dgn
    dong gitu harus hal hanya ini itu jadi jika juga kalau kami kamu karena kata ke kenapa kok kini lagi lah lain lalu lebih
    loh maka masih mau mereka meski mungkin nah namun nanti nya oleh pada para pernah pun saat saja sama sampai sangat
    sebagai sedang sejak sekarang semua seperti setelah sih sudah supaya tapi telah tentang terhadap tersebut tetapi
    untuk usai yaitu yang ya url user email phone wkwk haha tak tidak gak ga nggak enggak belum jangan bukan""".split())
with stage("preprocessing", len(U)):
    U["safe"] = apply(U.text, SAFE); U["embed"] = apply(U.safe, [replace_urls, replace_emails, replace_phones, replace_mentions, normalize_whitespace])
    U["normal"] = apply(U.embed, NORMAL); bow = apply(U.normal, CLEAN).str.split()
    df_bow = pd.Series(Counter(w for s in bow for w in set(s))) / len(bow); DOMAIN_STOP = set(df_bow[df_bow > DF_MAX].index)
    U["bow"] = bow.map(lambda s: " ".join(w for w in s if w not in STOP | DOMAIN_STOP and len(w) > 2))
with stage("stemming", len(U)):
    U["bow_stem"] = stem_series(U.bow)
show(f"Stopword domain · df > {DF_MAX:.0%}", df_bow[df_bow > DF_MAX].sort_values(ascending=False).rename("df"))
show("Audit preprocessing", pd.DataFrame({v: {"kosong_%_baris": 100 * W[U[v].str.strip().eq("").to_numpy()].sum() / W.sum(),
    "kata_median": U[v].str.split().str.len().median(), "kosakata": len(set(" ".join(U[v]).split()))} for v in ["safe", "normal", "bow", "bow_stem"]}).T)
rng = np.random.default_rng(SEED); S = np.sort(rng.choice(len(U), min(N_FIT, len(U)), replace=False))

[waktu] preprocessing: 20 detik | total 0.7 menit


stemming 12407 kata dari 32029 kata unik
[waktu] stemming: 312 detik | total 5.9 menit

Stopword domain · df > 25%


,df
di,0.289



Audit preprocessing


,kosong_%_baris,kata_median,kosakata
safe,0.0,9.0,57161.0
normal,0.0,9.0,56188.0
bow,0.0,7.0,32029.0
bow_stem,0.0,7.0,28814.0


## 05 · Fungsi evaluasi tanpa label

Fungsinya sama dengan versi review. Bedanya, ukuran topik dihitung berbobot `count`, dan semua fungsi menerima `idx`, yaitu baris `U` yang dinilai.

In [14]:
REF = {}
def reference(view):
    if view not in REF:
        vectorizer = CountVectorizer(binary=True, min_df=MIN_DF, token_pattern=r"\S+").fit(U[view])
        matrix = vectorizer.transform(U[view]).tocsc(); REF[view] = (matrix, vectorizer.vocabulary_, np.asarray(matrix.sum(0)).ravel())
    return REF[view]
def npmi(words, view):
    matrix, vocab, docfreq = reference(view); n = matrix.shape[0]; ids = [vocab[w] for w in words if w in vocab]; scores = []
    for a, b in itertools.combinations(ids, 2):
        both = matrix[:, a].multiply(matrix[:, b]).sum() / n
        scores.append(-1. if both == 0 else np.log(both / (docfreq[a] * docfreq[b] / n ** 2)) / -np.log(both))
    return float(np.mean(scores)) if scores else np.nan
def ctfidf(labels, view, idx, n=25):
    texts = U[view].to_numpy()[idx]; topics = [c for c in np.unique(labels) if c >= 0]
    joined = [" ".join(texts[labels == c]) for c in topics]
    vectorizer = CountVectorizer(token_pattern=r"\S+").fit(joined); tf = vectorizer.transform(joined).toarray().astype(float)
    weight = tf * np.log1p(tf.sum() / len(topics) / np.maximum(tf.sum(0), 1)); names = vectorizer.get_feature_names_out()
    return {c: list(names[np.argsort(-weight[i])[:n]]) for i, c in enumerate(topics)}
def match_stability(runs):
    scores = []
    for a, b in itertools.combinations(runs, 2):
        A, B = list(a.values()), list(b.values())
        sim = np.array([[len(set(x[:10]) & set(y[:10])) / len(set(x[:10]) | set(y[:10])) for y in B] for x in A])
        r, c = linear_sum_assignment(-sim); scores.append(sim[r, c].sum() / max(len(A), len(B)))
    return float(np.mean(scores))
def audit_scores(labels, idx):
    truth = U["_audit"].to_numpy()[idx]; w = W[idx]
    table = pd.crosstab(labels, truth, values=w, aggfunc="sum").fillna(0).to_numpy(); r, c = linear_sum_assignment(-table)
    return {"audit_NMI": normalized_mutual_info_score(np.repeat(truth, w), np.repeat(labels, w)), "audit_purity": table.max(1).sum() / w.sum(),
            "audit_acc_hungarian": table[r, c].sum() / w.sum()}
rows, STORE = [], {}
def evaluate(name, topics, labels, idx, view, **extra):
    labels = np.asarray(labels); valid = labels >= 0; w = W[idx]
    share = pd.Series(w[valid]).groupby(labels[valid]).sum() / w.sum()
    per_topic = {c: npmi(ws[:TOP], view) for c, ws in topics.items()}; top25 = [x for ws in topics.values() for x in ws[:25]]
    coherence = np.nanmean(list(per_topic.values())); diversity = len(set(top25)) / max(len(top25), 1)
    row = {"model": name, **extra, "n": len(idx), "k": len(topics), "outlier_%": 100 * w[~valid].sum() / w.sum(),
           "max_share_%": 100 * share.max() if len(share) else np.nan, "NPMI": coherence, "diversity": diversity, "TQ": (coherence + 1) / 2 * diversity}
    if AUDIT_COL: row.update(audit_scores(labels, idx))
    rows.append(row); STORE[name] = {"topics": topics, "labels": labels, "idx": idx, "view": view}; return row

## 06 · Representasi & ablasi di sampel

e5 dihitung sekali untuk semua teks unik, karena dibutuhkan model e5, penugasan outlier, dan analisis. Ablasi preprocessing memakai NMF di sampel `S`.

In [15]:
def locate(name, hf_id):
    found = list(INPUT.glob(f"**/models/{name}/config.json")); return str(found[0].parent) if found else hf_id
def embed(model_id, texts):
    from transformers import AutoTokenizer, AutoModel
    tokenizer = AutoTokenizer.from_pretrained(model_id); model = AutoModel.from_pretrained(model_id).cuda().half().eval()
    order = np.argsort([len(x) for x in texts]); parts = []
    with torch.inference_mode():
        for i in range(0, len(texts), BATCH):
            enc = tokenizer([texts[j] for j in order[i:i + BATCH]], padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to("cuda")
            hidden = model(**enc).last_hidden_state; mask = enc["attention_mask"].unsqueeze(-1).to(hidden.dtype)
            parts.append(((hidden * mask).sum(1) / mask.sum(1)).float().cpu().numpy())
    vectors = np.zeros((len(texts), parts[0].shape[1]), "float32"); vectors[order] = np.concatenate(parts)
    del model; torch.cuda.empty_cache(); return normalize(vectors)
with stage("embed_e5_penuh", len(U)):
    E5 = embed(locate("e5", "intfloat/multilingual-e5-base"), ("query: " + U.embed).tolist())
    E5C = normalize(E5 - E5.mean(0))   # dipusatkan: kemiripan antar topik tidak lagi semuanya ~0,99
def factor_topics(components, names, n=25): return {i: list(names[np.argsort(-row)[:n]]) for i, row in enumerate(components)}
def vectorizers(view):
    tfidf = TfidfVectorizer(min_df=MIN_DF, max_df=.5, token_pattern=r"\S+", sublinear_tf=True, max_features=50000).fit(U[view].iloc[S])
    count = CountVectorizer(min_df=MIN_DF, max_df=.5, token_pattern=r"\S+", max_features=50000).fit(U[view].iloc[S])
    return tfidf, count
K_REF = K_GRID[len(K_GRID) // 2]; ablation = []
with stage("ablasi", len(S)):
    for view in ["normal", "bow", "bow_stem"]:
        tfidf, _ = vectorizers(view); X = tfidf.transform(U[view].iloc[S])
        model = NMF(K_REF, init="nndsvda", max_iter=300, random_state=SEED); Wm = model.fit_transform(X)
        ablation.append({"view": view, **evaluate(f"ablasi_{view}", factor_topics(model.components_, tfidf.get_feature_names_out()),
                                                    np.where(Wm.max(1) > 0, Wm.argmax(1), -1), S, view, algo="ablasi")})
rows = [r for r in rows if r.get("algo") != "ablasi"]
ablation = pd.DataFrame(ablation).drop(columns=["model", "algo"]); show(f"6.1 Ablasi · NMF k={K_REF}, sampel {len(S)}", ablation)
BOW_VIEW = ablation.sort_values("TQ").iloc[-1]["view"]; print("Versi BoW terpilih:", BOW_VIEW)

[waktu] embed_e5_penuh: 83 detik | total 7.3 menit


[waktu] ablasi: 14 detik | total 7.5 menit

6.1 Ablasi · NMF k=20, sampel 30000


,view,n,k,outlier_%,max_share_%,NPMI,diversity,TQ,audit_NMI,audit_purity,audit_acc_hungarian
0,normal,30000,20,0.027,15.906,0.024,0.612,0.313,0.051,0.382,0.164
1,bow,30000,20,0.043,9.454,0.011,0.710,0.359,0.099,0.427,0.181
2,bow_stem,30000,20,0.037,12.645,0.044,0.718,0.375,0.095,0.420,0.177


Versi BoW terpilih: bow_stem


## 07 · Grid di sampel

Lima model × `K_GRID` × 3 seed di sampel `S`. HDBSCAN memvariasikan `min_cluster_size`. Pembanding acak dibuat dari kata acak.

In [16]:
TFIDF, COUNT = vectorizers(BOW_VIEW)
X_t, X_c = TFIDF.transform(U[BOW_VIEW].iloc[S]), COUNT.transform(U[BOW_VIEW].iloc[S])
names_t, names_c = TFIDF.get_feature_names_out(), COUNT.get_feature_names_out(); MODELS = {}
def fit_once(algo, k, seed):
    if algo == "tfidf_kmeans":
        m = KMeans(k, n_init=1, random_state=seed).fit(X_t); return ctfidf(m.labels_, BOW_VIEW, S), m.labels_, m
    if algo == "e5_kmeans":
        m = KMeans(k, n_init=1, random_state=seed).fit(E5[S]); return ctfidf(m.labels_, BOW_VIEW, S), m.labels_, m
    if algo == "lda":
        m = LatentDirichletAllocation(k, learning_method="online", batch_size=2048, max_iter=10, random_state=seed); theta = m.fit_transform(X_c)
        return factor_topics(m.components_, names_c), theta.argmax(1), m
    m = NMF(k, init="nndsvda" if seed == SEED else "random", max_iter=300, random_state=seed); Wm = m.fit_transform(X_t)
    return factor_topics(m.components_, names_t), np.where(Wm.max(1) > 0, Wm.argmax(1), -1), m
with stage("grid_sampel", len(S)):
    for algo in ["tfidf_kmeans", "lda", "nmf", "e5_kmeans"]:
        for k in K_GRID:
            fits = [fit_once(algo, k, SEED + s) for s in range(SEEDS)]
            evaluate(f"{algo}_k{k}", fits[0][0], fits[0][1], S, BOW_VIEW, algo=algo, stability=match_stability([f[0] for f in fits]))
            MODELS[f"{algo}_k{k}"] = fits[0][2]

[waktu] grid_sampel: 798 detik | total 20.9 menit


In [17]:
import umap
with stage("bertopic_sampel", len(S)):
    reducers = [umap.UMAP(n_neighbors=15, n_components=5, min_dist=0., metric="cosine", random_state=SEED + s, low_memory=True).fit(E5[S]) for s in range(SEEDS)]
    for size in [len(S) // 400, len(S) // 200, len(S) // 100]:
        fits = [HDBSCAN(min_cluster_size=size, min_samples=10).fit_predict(r.embedding_) for r in reducers]
        topics = [ctfidf(f, BOW_VIEW, S) for f in fits]
        evaluate(f"bertopic_mcs{size}", topics[0], fits[0], S, BOW_VIEW, algo="bertopic", stability=match_stability(topics), min_cluster_size=size)
        MODELS[f"bertopic_mcs{size}"] = fits[0]
rand = np.random.default_rng(SEED)
evaluate("acak_k10", {i: list(rand.choice(names_t, 25, replace=False)) for i in range(10)}, rand.integers(0, 10, len(S)), S, BOW_VIEW, algo="acak", stability=0.)
board = pd.DataFrame(rows)
show("Grid · tanpa audit", board.drop(columns=[c for c in board if c.startswith("audit_")]).sort_values("TQ", ascending=False).head(12))

I0000 00:00:1794168664.578595      46 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


[waktu] bertopic_sampel: 180 detik | total 24.4 menit

Grid · tanpa audit


,model,algo,stability,n,k,outlier_%,max_share_%,NPMI,diversity,TQ,min_cluster_size
12,nmf_k8,nmf,0.558,30000,8,0.037,30.934,0.093,0.810,0.443,NaN
26,bertopic_mcs300,bertopic,0.747,30000,20,40.228,8.266,0.055,0.812,0.428,300.0
13,nmf_k10,nmf,0.680,30000,10,0.037,18.891,0.082,0.784,0.424,NaN
6,lda_k8,lda,0.215,30000,8,0.000,21.227,-0.003,0.850,0.424,NaN
25,bertopic_mcs150,bertopic,0.660,30000,33,36.764,8.266,0.083,0.777,0.421,150.0
7,lda_k10,lda,0.194,30000,10,0.000,14.592,-0.013,0.852,0.420,NaN
18,e5_kmeans_k8,e5_kmeans,0.657,30000,8,0.000,17.484,0.078,0.770,0.415,NaN
19,e5_kmeans_k10,e5_kmeans,0.780,30000,10,0.000,15.164,0.066,0.764,0.407,NaN
8,lda_k15,lda,0.150,30000,15,0.000,13.121,-0.090,0.883,0.402,NaN
9,lda_k20,lda,0.140,30000,20,0.000,11.088,-0.095,0.870,0.394,NaN


## 08 · Pemilihan tanpa label & C_V

Larangan topik raksasa dan outlier berlebihan, lalu rata-rata peringkat TQ dan stabilitas. C_V (gensim) dihitung di sampel untuk 5 kandidat teratas, sebagai laporan saja.

In [18]:
pool = board[(board.algo != "acak") & (board["max_share_%"] <= 100 * MAX_SHARE) & (board["outlier_%"] <= 100 * MAX_OUTLIER)].copy()
pool = pool if len(pool) else board[board.algo != "acak"].copy()
pool["skor_gabungan"] = pool[["TQ", "stability"]].rank(pct=True).mean(axis=1)
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
with stage("c_v", len(S)):
    tokens = U[BOW_VIEW].iloc[S].str.split().tolist(); dictionary = Dictionary(tokens)
    def c_v(name):
        topics = [[w for w in ws[:TOP] if w in dictionary.token2id] for ws in STORE[name]["topics"].values()]
        return CoherenceModel(topics=[t for t in topics if len(t) >= 2], texts=tokens, dictionary=dictionary, coherence="c_v", processes=1).get_coherence()
    top5 = pool.sort_values("skor_gabungan", ascending=False).head(5).copy(); top5["C_V"] = [c_v(m) for m in top5.model]
show("5 kandidat teratas", top5[["model", "algo", "k", "TQ", "NPMI", "diversity", "stability", "C_V", "max_share_%", "outlier_%", "skor_gabungan"]])
FINAL_BASE = top5.iloc[0].model; ALGO = top5.iloc[0].algo; print("TERPILIH:", FINAL_BASE)

[waktu] c_v: 5 detik | total 24.9 menit

5 kandidat teratas


,model,algo,k,TQ,NPMI,diversity,stability,C_V,max_share_%,outlier_%,skor_gabungan
19,e5_kmeans_k10,e5_kmeans,10,0.407,0.066,0.764,0.780,0.472,15.164,0.000,0.861
13,nmf_k10,nmf,10,0.424,0.082,0.784,0.680,0.493,18.891,0.037,0.833
18,e5_kmeans_k8,e5_kmeans,8,0.415,0.078,0.770,0.657,0.475,17.484,0.000,0.722
12,nmf_k8,nmf,8,0.443,0.093,0.810,0.558,0.521,30.934,0.037,0.694
14,nmf_k15,nmf,15,0.388,0.048,0.741,0.698,0.464,20.375,0.037,0.667


TERPILIH: e5_kmeans_k10


## 09 · Terapkan ke semua teks unik & post-processing

Model dari sampel diterapkan ke semua teks: `transform` untuk LDA/NMF, `predict` untuk KMeans, dan centroid e5 untuk HDBSCAN. **Cek transfer:** di baris sampel, topik hasil penerapan dibandingkan dengan topik saat fit. Lalu post-processing seperti versi review:
- outlier ke topik terdekat
- gabung topik kembar (Jaccard kata kunci ≥ `MERGE_JACCARD`)
- gabung topik di bawah `MIN_TOPIC_SHARE`
- label ulang dengan c-TF-IDF dari semua anggota

In [19]:
ALL = np.arange(len(U)); sample_labels = STORE[FINAL_BASE]["labels"]
def nearest_topic(labels_known, idx_known, idx_target):
    keep = [c for c in np.unique(labels_known) if c >= 0]
    centers = normalize(np.vstack([E5[idx_known][labels_known == c].mean(0) for c in keep]))
    return np.array(keep)[(E5[idx_target] @ centers.T).argmax(1)]
with stage("terapkan_penuh", len(U)):
    model = MODELS[FINAL_BASE]; text_all = U[BOW_VIEW]
    if ALGO == "nmf": Wm = model.transform(TFIDF.transform(text_all)); labels = np.where(Wm.max(1) > 0, Wm.argmax(1), -1)
    elif ALGO == "lda": labels = model.transform(COUNT.transform(text_all)).argmax(1)
    elif ALGO == "tfidf_kmeans": labels = model.predict(TFIDF.transform(text_all))
    elif ALGO == "e5_kmeans": labels = model.predict(E5)
    else:
        labels = np.full(len(U), -1); labels[S] = sample_labels
        rest = np.setdiff1d(ALL, S); labels[rest] = nearest_topic(sample_labels, S, rest)
transfer = adjusted_rand_score(sample_labels, labels[S])
print(f"Transfer sampel → penuh: ARI {transfer:.3f} di {len(S)} teks sampel")

[waktu] terapkan_penuh: 0 detik | total 24.9 menit
Transfer sampel → penuh: ARI 1.000 di 30000 teks sampel


In [20]:
with stage("post_processing", len(U)):
    steps = []
    if (labels < 0).any():
        bad = np.where(labels < 0)[0]; good = np.where(labels >= 0)[0]
        labels = labels.copy(); labels[bad] = nearest_topic(labels[good], good, bad); steps.append(f"{len(bad)} outlier")
    topics_now = ctfidf(labels, BOW_VIEW, ALL); merged = []
    for a, b in itertools.combinations(sorted(topics_now), 2):
        A, B = set(topics_now[a][:10]), set(topics_now[b][:10])
        if len(A & B) / len(A | B) >= MERGE_JACCARD: merged.append((a, b))
    for a, b in merged:
        small, large = sorted([a, b], key=lambda c: W[labels == c].sum()); labels[labels == small] = large
    if merged: steps.append(f"gabung {len(merged)} pasangan kembar")
    share = pd.Series(W).groupby(labels).sum() / W.sum(); tiny = share[share < MIN_TOPIC_SHARE].index
    if len(tiny):
        moving = np.where(np.isin(labels, tiny))[0]; staying = np.where(~np.isin(labels, tiny))[0]
        labels[moving] = nearest_topic(labels[staying], staying, moving); steps.append(f"gabung {len(tiny)} topik kecil")
    labels = pd.Series(labels).map({c: i for i, c in enumerate(sorted(np.unique(labels)))}).to_numpy()
    final_topics = ctfidf(labels, BOW_VIEW, ALL)
    evaluate("FINAL_penuh", final_topics, labels, ALL, BOW_VIEW, algo="final")
board = pd.DataFrame(rows)
show("Sebelum dan sesudah · tanpa audit", board[board.model.isin([FINAL_BASE, "FINAL_penuh"])].drop(columns=[c for c in board if c.startswith("audit_")]))
print("Langkah:", steps or "tidak ada")

[waktu] post_processing: 2 detik | total 24.9 menit

Sebelum dan sesudah · tanpa audit


,model,algo,stability,n,k,outlier_%,max_share_%,NPMI,diversity,TQ,min_cluster_size
19,e5_kmeans_k10,e5_kmeans,0.78,30000,10,0.0,15.164,0.066,0.764,0.407,NaN
28,FINAL_penuh,final,NaN,90885,10,0.0,15.307,0.071,0.752,0.403,NaN


Langkah: tidak ada


## 10 · Evaluasi & kelemahan

Kolom audit dibuka di sini kalau ada, tanpa mengubah keputusan di atas. Kemiripan antar topik dihitung di ruang e5 yang sudah dipusatkan. Topik raksasa yang kata kuncinya generik ditandai sebagai **topik sampah**.

In [21]:
audit_cols = [c for c in board if c.startswith("audit_")]
show("10.1 Semua kandidat", board.sort_values(audit_cols[0] if audit_cols else "TQ", ascending=False).head(20))
if AUDIT_COL:
    grid = board[board.algo.isin(["tfidf_kmeans", "lda", "nmf", "e5_kmeans", "bertopic"])]
    final_nmi = board.loc[board.model.eq("FINAL_penuh"), "audit_NMI"].iloc[0]
    print(f"FINAL_penuh: NMI {final_nmi:.3f} | terbaik di grid sampel: {grid.sort_values('audit_NMI').iloc[-1].model} NMI {grid.audit_NMI.max():.3f}")
    trust = pd.DataFrame({m: spearmanr(grid[m], grid.audit_NMI, nan_policy="omit") for m in ["TQ", "NPMI", "diversity", "stability", "max_share_%", "k"]}, index=["spearman", "p"]).T
    show("10.2 Kriteria tanpa label vs NMI audit", trust.sort_values("spearman", ascending=False))
    show("NMI terbaik per model", grid.groupby("algo").audit_NMI.max().sort_values(ascending=False))


10.1 Semua kandidat


,model,algo,stability,n,k,outlier_%,max_share_%,NPMI,diversity,TQ,audit_NMI,audit_purity,audit_acc_hungarian,min_cluster_size
24,bertopic_mcs75,bertopic,0.668,30000,58,32.575,11.427,0.055,0.731,0.386,0.346,0.677,0.386,75.0
26,bertopic_mcs300,bertopic,0.747,30000,20,40.228,8.266,0.055,0.812,0.428,0.336,0.604,0.391,300.0
25,bertopic_mcs150,bertopic,0.660,30000,33,36.764,8.266,0.083,0.777,0.421,0.333,0.635,0.370,150.0
23,e5_kmeans_k30,e5_kmeans,0.632,30000,30,0.000,5.760,0.046,0.673,0.352,0.282,0.631,0.232,NaN
22,e5_kmeans_k25,e5_kmeans,0.584,30000,25,0.000,6.795,0.051,0.696,0.366,0.272,0.614,0.241,NaN
21,e5_kmeans_k20,e5_kmeans,0.580,30000,20,0.000,8.908,0.050,0.698,0.367,0.259,0.583,0.259,NaN
20,e5_kmeans_k15,e5_kmeans,0.567,30000,15,0.000,10.256,0.059,0.683,0.361,0.245,0.570,0.300,NaN
18,e5_kmeans_k8,e5_kmeans,0.657,30000,8,0.000,17.484,0.078,0.770,0.415,0.240,0.520,0.346,NaN
19,e5_kmeans_k10,e5_kmeans,0.780,30000,10,0.000,15.164,0.066,0.764,0.407,0.239,0.529,0.341,NaN
28,FINAL_penuh,final,NaN,90885,10,0.000,15.307,0.071,0.752,0.403,0.237,0.529,0.339,NaN


FINAL_penuh: NMI 0.237 | terbaik di grid sampel: bertopic_mcs75 NMI 0.346

10.2 Kriteria tanpa label vs NMI audit


,spearman,p
NPMI,0.667,0.000
stability,0.617,0.001
TQ,0.313,0.112
k,0.275,0.165
diversity,-0.056,0.783
max_share_%,-0.571,0.002



NMI terbaik per model


,audit_NMI
algo,
bertopic,0.346
e5_kmeans,0.282
lda,0.096
nmf,0.095
tfidf_kmeans,0.080


In [22]:
matrix, vocab, docfreq = reference(BOW_VIEW); generic = {w for w, i in vocab.items() if docfreq[i] / matrix.shape[0] > .03}
order = sorted(final_topics); centers = normalize(np.vstack([E5C[labels == c].mean(0) for c in order]))
cosine = (E5C * centers[labels]).sum(1)
cards = []
for c in order:
    m = labels == c; w = W[m]; ws = final_topics[c]
    row = {"topik": c, "label": " / ".join(ws[:3]), "baris": int(w.sum()), "%_baris": 100 * w.sum() / W.sum(), "teks_unik": int(m.sum()),
           "kata_kunci": ", ".join(ws[:TOP]), "NPMI": npmi(ws[:TOP], BOW_VIEW), "kata_generik_%": 100 * np.mean([x in generic for x in ws[:TOP]]),
           "porsi_teks_terbanyak_%": 100 * w.max() / w.sum(), "kemiripan_ke_pusat": cosine[m].mean(),
           "contoh": " | ".join(U.text[m].iloc[np.argsort(-cosine[m])[:2]].str.slice(0, 60))}
    if DESC_COL: row[f"rata_{DESC_COL}"] = np.average(U.desc[m], weights=w)
    if AUDIT_COL:
        audit = pd.Series(w).groupby(U["_audit"].to_numpy()[m]).sum().sort_values(ascending=False)
        row.update(audit_mayoritas=audit.index[0], audit_purity=audit.iloc[0] / w.sum())
    cards.append(row)
cards = pd.DataFrame(cards)
show("10.3 Kartu topik · urut dari NPMI terendah", cards.sort_values("NPMI"))
flags = pd.DataFrame({"topik": cards.topik, "label": cards.label, "topik_sampah": (cards["%_baris"] > 100 * GIANT_GENERIC) & (cards["kata_generik_%"] >= 30),
    "NPMI_negatif": cards.NPMI < 0, "didominasi_satu_teks": cards["porsi_teks_terbanyak_%"] > 30, "kecil": cards["%_baris"] < 1})
show("10.4 Bendera kelemahan", flags[flags.drop(columns=["topik", "label"]).any(axis=1)])


10.3 Kartu topik · urut dari NPMI terendah


,topik,label,baris,%_baris,teks_unik,kata_kunci,NPMI,kata_generik_%,porsi_teks_terbanyak_%,kemiripan_ke_pusat,contoh,audit_mayoritas,audit_purity
5,5,banjir / hujan / jakarta,3867,4.249,3856,"banjir, hujan, jakarta, gempa, warga, rendam, korban, air, bmkg, jalan",-0.029,0.0,0.129,0.375,"11 Jalan di Jakarta Terendam Banjir, Ketinggian Air 5-30 Cm | Banjir Jakarta, Underpass Kemayora...",news,0.703
4,4,tinggal / anak / nikah,13129,14.425,13111,"tinggal, anak, nikah, film, the, tahun, foto, istri, lagu, cerita",-0.005,0.0,0.053,0.274,"Suka Nyanyi Sejak Kecil, Hanin Dhiya Ungkap Kekagumannya Pad | Sosok Ini Dicintai hingga Diangga...",hot,0.752
0,0,foto / potret / dunia,9452,10.385,9428,"foto, potret, dunia, lihat, indah, bikin, cara, cantik, hari, kisah",-0.003,0.0,0.042,0.307,Keindahan Pahawang yang Membekas di Hati | Terpukau Keindahan Rahasia Langit Malam,travel,0.262
2,2,soal / kerja / mudik,11251,12.361,11238,"soal, kerja, mudik, buka, bakal, minta, larang, usaha, jiwasraya, jokowi",0.012,20.0,0.027,0.282,"Bahlil Janji Mundur Jika Gagal Genjot Kemudahan Berusaha | Utang Usaha ke Bank Boleh Nunggak, OJ...",finance,0.359
3,3,harga / juta / indonesia,10752,11.813,10726,"harga, juta, indonesia, mobil, jakarta, buka, mulai, naik, listrik, siap",0.036,0.0,0.028,0.226,Pilihan Mobil yang Bisa Dibeli dengan Gaji Rp 3-4 Jutaan | Mobil Baru di Indonesia Tahun 2019 Ju...,finance,0.331
6,6,jokowi / kpk / minta,9733,10.694,9730,"jokowi, kpk, minta, dpr, dki, anies, soal, psbb, perintah, ketua",0.081,20.0,0.021,0.267,"PKS Minta KPK Awasi Proses Pemilihan Cawagub DKI di DPRD | Pemerintah Pusat Soroti Bansos DKI, P...",news,0.728
1,1,corona / virus / pandemi,13932,15.307,13918,"corona, virus, pandemi, covid, cegah, tengah, wabah, masker, imbas, dampak",0.093,20.0,0.014,0.315,Bahaya! Virus Corona Bisa Buat Ekonomi Lumpuh | Cara Mencegah Virus Corona,news,0.307
8,8,makan / enak / masak,5211,5.725,5207,"makan, enak, masak, bikin, rumah, resep, cara, hingga, tips, galeri",0.116,10.0,0.038,0.352,"Sarapan di Rumah Lebih Bernutrisi dengan 5 Menu Makanan Prak | Agar Tetap Seru di Rumah, 5 Kuis ...",food,0.611
9,9,polisi / tangkap / tewas,7815,8.586,7814,"polisi, tangkap, tewas, pria, warga, bunuh, duga, viral, laku, orang",0.154,10.0,0.026,0.315,"Pencuri Motor yang Pukul Korbannya Pakai Sajam Ditangkap Pol | Panah Warga hingga Tewas, 2 Pria ...",news,0.843
7,7,corona / positif / pasien,5875,6.455,5857,"corona, positif, pasien, kasus, sembuh, tinggal, orang, tambah, indonesia, virus",0.254,20.0,0.034,0.440,"Kasus Pasien Positif Virus Corona Jadi 34: 2 Sembuh, 1 Menin | Update Terbaru Virus Corona di In...",news,0.733



10.4 Bendera kelemahan


,topik,label,topik_sampah,NPMI_negatif,didominasi_satu_teks,kecil
0,0,foto / potret / dunia,False,True,False,False
4,4,tinggal / anak / nikah,False,True,False,False
5,5,banjir / hujan / jakarta,False,True,False,False


In [23]:
sim = centers @ centers.T; np.fill_diagonal(sim, -1)
close = sorted([(a, b, sim[a, b]) for a in range(len(order)) for b in range(a + 1, len(order))], key=lambda x: -x[2])[:6]
show("10.5 Pasangan topik paling mirip · e5 dipusatkan", pd.DataFrame(close, columns=["topik_a", "topik_b", "cosine"]).assign(
    label_a=lambda f: f.topik_a.map(cards.set_index("topik").label), label_b=lambda f: f.topik_b.map(cards.set_index("topik").label)))
length = pd.cut(U[BOW_VIEW].str.split().str.len(), [-1, 0, 2, 5, 12, np.inf], labels=["kosong", "1-2", "3-5", "6-12", ">12"])
slices = pd.DataFrame({"baris_%": pd.Series(W).groupby(length.to_numpy(), observed=True).sum() / W.sum() * 100,
                       "kemiripan_ke_pusat": pd.Series(cosine).groupby(length.to_numpy(), observed=True).mean()})
if AUDIT_COL:
    truth = U["_audit"].to_numpy()
    slices["NMI_slice"] = [normalized_mutual_info_score(np.repeat(truth[length.to_numpy() == v], W[length.to_numpy() == v]),
                                                        np.repeat(labels[length.to_numpy() == v], W[length.to_numpy() == v])) for v in slices.index]
show("10.6 Kelemahan menurut panjang dokumen", slices)
show("10.7 Paling jauh dari pusat topik", U.assign(topik=labels, kemiripan=cosine).nsmallest(6, "kemiripan")[["text", "count", "topik", "kemiripan"]])


10.5 Pasangan topik paling mirip · e5 dipusatkan


,topik_a,topik_b,cosine,label_a,label_b
0,1,7,0.578,corona / virus / pandemi,corona / positif / pasien
1,0,2,0.408,foto / potret / dunia,soal / kerja / mudik
2,5,9,0.212,banjir / hujan / jakarta,polisi / tangkap / tewas
3,4,8,0.166,tinggal / anak / nikah,makan / enak / masak
4,2,6,0.137,soal / kerja / mudik,jokowi / kpk / minta
5,0,8,0.130,foto / potret / dunia,makan / enak / masak



10.6 Kelemahan menurut panjang dokumen


,baris_%,kemiripan_ke_pusat,NMI_slice
1-2,0.202,0.307,0.227
3-5,12.130,0.305,0.181
6-12,87.602,0.301,0.239
>12,0.066,0.330,0.391



10.7 Paling jauh dari pusat topik


,text,count,topik,kemiripan
80638,Salut! Ambulans Pelat Hitam Ini Hasil Gotong-Royong Warga,1,3,-0.032
68507,"Senangnya Warga Seko: Dulu ke Ibu Kota Luwu Utara 3 Malam, Kini 6 Jam",1,3,0.004
32093,Sarwendah Donor ASI ke Beberapa RS di Jakarta,1,3,0.007
89746,"99% Penerbangan Terhenti, Susi Air Rumahkan dan PHK Karyawan",1,3,0.007
21530,Dilema Anak Belajar di Rumah Saat Ortu Kembali Masuk Kerja,1,3,0.008
60891,"Usai Observasi di Natuna, Mahasiswi Sultra Lanjut Lanjut Kuliah Online",1,3,0.012


## 11 · Keluaran & waktu

`topics.csv` berisi topik dan label untuk setiap baris asli.

In [24]:
label_text = cards.set_index("topik").label
data["topik"] = data.key.map(dict(zip(U.key, labels))); assert data.topik.notna().all()
data[["row_id", TEXT_COL, "topik"]].assign(label=data.topik.map(label_text)).to_csv(OUT / "topics.csv", index=False)
board.to_csv(OUT / "kandidat.csv", index=False); cards.to_csv(OUT / "kartu_topik.csv", index=False)
times = pd.DataFrame(TIMES); times["menit"] = times.detik / 60; show("Waktu per tahap", times)
total = (time.perf_counter() - RUN_START) / 60
import importlib.metadata as metadata
manifest = {"dataset": DATASET, "rows": len(data), "unique": len(U), "n_fit": len(S), "bow_view": BOW_VIEW, "base": FINAL_BASE,
    "k_final": len(final_topics), "transfer_ARI": transfer, "post": steps, "minutes": total, "times": TIMES,
    "final": {k: v for k, v in board.loc[board.model.eq("FINAL_penuh")].iloc[0].items() if isinstance(v, (int, float, np.floating))},
    "gpu": torch.cuda.get_device_name(0), "versions": {p: metadata.version(p) for p in ["torch", "transformers", "scikit-learn", "umap-learn", "gensim"]}}
(OUT / "run_manifest.json").write_text(json.dumps(manifest, indent=2, default=float))
print(f"Total {total:.1f} menit | batas {BUDGET_MINUTES} menit"); assert total < BUDGET_MINUTES


Waktu per tahap


,tahap,detik,n,menit
0,muat_data,3.427,NaN,0.057
1,eda,2.397,90885.0,0.040
2,preprocessing,19.809,90885.0,0.330
3,stemming,311.901,90885.0,5.198
4,embed_e5_penuh,83.395,90885.0,1.390
5,ablasi,14.489,30000.0,0.241
6,grid_sampel,798.367,30000.0,13.306
7,bertopic_sampel,179.808,30000.0,2.997
8,c_v,5.003,30000.0,0.083
9,terapkan_penuh,0.059,90885.0,0.001


Total 25.0 menit | batas 290 menit


## Batas interpretasi
- Pilihan dibuat di sampel. Bagian 09 mengukur seberapa sama hasil sampel dan hasil penerapan penuh.
- Kategori berita bukan topik. NMI audit hanya sinyal kasar: topik koheren yang lintas kategori, misalnya COVID, justru menurunkan NMI.
- Stopword dan stopword domain bergantung data, jadi periksa tabel di bagian 04.